# Lesson 2 — Design of experiments & batch data generation

*Part of the [PlugFlowML course](../README.md): machine-learning surrogates for physical simulation, taught on a chemical reactor.*

**Mode:** read-along (requires a kinetic mechanism — outputs shown below) · **Runtime:** if executed, minutes for the 10-run demo config, up to many hours for a full campaign (~2 min per simulation) · **Builds on:** Lesson 1

**What you'll learn**

1. Choose a sampling design for an expensive simulator — Latin Hypercube vs grid vs random — and justify the choice in terms of coverage per run.
2. Turn a single simulation into a batch campaign: parameter bounds, run budget, incremental saves, and metadata for reproducibility.
3. Treat failed runs as a normal, informative part of a simulation campaign, and know what to check before training on the survivors.
4. Recognise an embarrassingly parallel workload and how such campaigns are chunked across an HPC cluster (the SLURM-array pattern).

**The example system.** This notebook wraps Lesson 1's single PFR run in a loop. Six inlet/geometry parameters — feed temperature, pressure, tube length and diameter, mass flow rate, and wall heat flux — are sampled within configured bounds; each sample becomes one Cantera simulation, and each simulation contributes ~200 axial rows of 245+ features, saved incrementally with a metadata record for reproducibility. The dataset used in the rest of the course came from exactly this notebook run at scale: 50,000 n-hexane simulations on an HPC cluster (~92% of which succeeded), yielding ~9.2 million rows.

> ⚠️ **Read-along lesson.** This notebook needs a detailed kinetic mechanism
> that we cannot redistribute. All outputs are committed, so you can follow
> every step without running it. If you have your own Cantera mechanism, point
> `configs/simulation/main1_reactant_database.json` at it and the notebook runs
> end-to-end. Hands-on work starts in [Lesson 3](Main_3_data_exploration_feature_engineering.ipynb).

In [ ]:
# Setup and Imports
import sys
import os
from pathlib import Path
import json
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import time
from datetime import datetime

# Project root: notebooks live under notebooks/; from there, project root is one level up
current_dir = Path(os.getcwd())
if (current_dir / 'src').exists():
    project_root = current_dir
elif (current_dir.parent / 'src').exists():
    project_root = current_dir.parent
else:
    project_root = current_dir
os.chdir(project_root)  # so configs/, data/training/, mechanisms/ resolve correctly

# IMPORTANT: Import cantera BEFORE adding src to sys.path
# This prevents namespace conflict: without this, Python would find src/cantera/
# (our package) instead of the actual cantera library
import cantera as ct
print(f"Cantera version: {ct.__version__}")

# Suppress all Cantera/SUNDIALS warnings and messages
import warnings
import logging

# Add project root to path (after cantera is imported)
sys.path.insert(0, str(project_root))

from src.ml.data_generation import TrainingDataGenerator

print("PlugFlowML: ML Training Data Generation")
print("=" * 60)
print(f"Project root: {project_root}")

## Step 1: Configuration

Configure the data generation parameters. You can either:
- Load from a JSON config file
- Set parameters directly in the notebook

Sampling method: `latin` (LHS), `random`, or `full_grid` / `structured_grid` / `grid`. Use `parameter_ranges` for grid; use `random_sample_bounds` for random/LHS.

In [ ]:
# Run control flags (metadata/data only)
IF_SAVE_METADATA = True  # Save metadata JSON when generating dataset
IF_SAVE_TRAINING_DATA = True  # Save training data (partial + final .pkl) to output_dir
IF_SAVE_COMPLETE_CSV = True   # Also write training_data_complete_*.csv (large); optional

# Load configuration from JSON (edit configs/ml/main2_data_generation_config.json)
CONFIG_FILE = 'configs/ml/main2_data_generation_config.json'

if not Path(CONFIG_FILE).exists():
    raise FileNotFoundError(f"Config not found: {CONFIG_FILE}")

with open(CONFIG_FILE, 'r') as f:
    config = json.load(f)
print(f"[OK] Loaded configuration from: {CONFIG_FILE}")

REACTANTS = config.get('reactants', ['ethane'])
MAX_COMBINATIONS = config.get('max_combinations_per_reactant', 100)
OUTPUT_DIR = config.get('output_dir', 'data/training')
SAVE_INTERVAL = config.get('save_interval', 10)
PARAM_RANGES = config.get('parameter_ranges', {})
SAMPLING_METHOD = config.get('sampling_method', 'latin_hypercube')
LHS_SEED = config.get('lhs_seed', 42)
RANDOM_SAMPLE_BOUNDS = config.get('random_sample_bounds', None)

print(f"  - Reactants: {REACTANTS}")
print(f"  - Max combinations per reactant: {MAX_COMBINATIONS}")
print(f"  - Output directory: {OUTPUT_DIR}")
print(f"  - Sampling method: {SAMPLING_METHOD}")

print(f"\nParameter Ranges:")
for param, values in PARAM_RANGES.items():
    if isinstance(values, list) and len(values) == 3:
        print(f"  - {param}: {values[0]} to {values[1]} ({values[2]} points)")
    else:
        print(f"  - {param}: {values}")

> 🧠 **Concept — Design of experiments: grid vs Latin Hypercube.** When every sample costs a full simulation, *where* you place your samples is a first-class design decision. A full grid with k points per parameter needs k^d runs: with the d = 6 parameters here, even a coarse 10-point grid means 10⁶ simulations, and the budget grows exponentially with every parameter you add. Worse, a grid spends that budget inefficiently — projected onto any single parameter, a million-run grid still visits only 10 distinct values. Latin Hypercube Sampling (LHS) inverts the deal: you fix the budget N first; the method slices each parameter's range into N equal strata, places exactly one sample in each, then shuffles the pairings between dimensions so the points also spread through the joint space. Every run then carries fresh information in *every* dimension at once — N runs give N distinct values of each parameter. Plain random sampling shares the fixed-budget property but leaves clumps and holes; LHS's stratification guarantees uniform 1D coverage. (Lesson 3 visualises the coverage this produces.)
>
> *In your domain:* the same trade appears in CFD parameter sweeps, perturbed-physics climate ensembles, and even ML hyperparameter search — random/LHS beats grid there for exactly this reason.

## Step 2: Initialize Data Generator

Create the data generator with your configuration.

In [ ]:
# Initialize generator
generator = TrainingDataGenerator(output_dir=OUTPUT_DIR, disable_plots=True)

# Update parameter ranges from config
if PARAM_RANGES:
    for key, value in PARAM_RANGES.items():
        if isinstance(value, list) and len(value) == 3:
            # Convert [min, max, n_points] to numpy array
            generator.param_ranges[key] = np.linspace(value[0], value[1], value[2])
            print(f"[OK] Updated {key}: {len(generator.param_ranges[key])} points")

# Calculate total combinations
total_combinations = generator._calculate_total_combinations()
print(f"\n[OK] Data generator initialized")
print(f"  - Output directory: {generator.output_dir}")
print(f"  - Total possible combinations: {total_combinations:,}")
print(f"  - Will generate: {len(REACTANTS) * MAX_COMBINATIONS:,} simulations")

#### 🤔 Think it through — six knobs, one budget

The cell above contrasts the total possible grid combinations (10 points per parameter over 6 parameters → 10⁶ per reactant; the printed total also multiplies across the 4 reactants in the database) with the number of simulations we will actually run. Why is an LHS design with a few thousand runs a defensible substitute for a million-run grid — and what does a grid offer that LHS cannot?

<details><summary><b>💡 Discussion</b></summary>

At ~2 minutes per simulation, 10⁶ runs is roughly four CPU-*years*, so the grid is unaffordable outright — but the deeper argument is information per run. A 10-points-per-axis grid explores only 10 distinct values of each parameter regardless of how many runs it costs, and in six dimensions almost all of its points sit deep in corners of the hypercube. Most physical responses are dominated by main effects and low-order interactions, which LHS estimates far more efficiently because every run contributes a fresh value in every dimension. What you give up: grids make certain analyses trivial (hold-everything-else-fixed slices, factorial ANOVA) and refine predictably, whereas an LHS design is not nested — you cannot extend it by simply "adding a few more points per axis"; you draw a new design or use a sequential space-filling scheme. For surrogate training, coverage per run wins.
</details>

### Step 2.1: Sampling preview moved to Main_3

EDA and training-space coverage plotting are handled in `Main_3_data_exploration_feature_engineering.ipynb`.

Preview how the parameter space will be explored: 1D marginals (uniformity) and 2D pairs (space filling).

In [ ]:
print("[INFO] Sampling preview removed from Main_2 for cleaner generation workflow.")
print("       Use Main_3_data_exploration_feature_engineering.ipynb for all EDA/coverage plots.")

## Step 3: Generate Training Data

Run the data generation process. This will:
- Run multiple PFR simulations with varied parameters
- Collect features and targets at each simulation point
- Save data incrementally to prevent loss
- Generate metadata for reproducibility

Note: This may take 5-30 minutes depending on the number of simulations.

> 🧠 **Concept — Embarrassingly parallel campaigns.** No simulation in this sweep depends on any other, which makes the campaign *embarrassingly parallel*: split the sampled conditions into chunks, run the chunks on whatever hardware is available — local cores, cluster nodes, cloud instances — and concatenate the results. This is precisely the SLURM job-array pattern on HPC systems, and it is how the course dataset was produced: 32 independent array tasks, each writing its own `training_data_complete_*.pkl`, consolidated afterwards into a single file with a merged metadata record. Two habits make campaigns like this safe at scale. Checkpointing: `save_interval` writes partial results every few runs, so a crash or a wall-time kill loses minutes of work, not hours. Metadata: the sampling method, seed, bounds and per-task provenance are written next to the data, so the campaign can be audited, reproduced or extended months later without archaeology.

In [ ]:
# Generate dataset
print("=" * 60)
print("Starting Training Data Generation...")
print("=" * 60)
print(f"Reactants: {REACTANTS}")
print(f"Max combinations per reactant: {MAX_COMBINATIONS}")
print(f"Sampling method: {SAMPLING_METHOD}")
print(f"Save interval: {SAVE_INTERVAL} simulations")
print("=" * 60)

start_time = time.time()

dataset = generator.generate_dataset(
    reactants=REACTANTS,
    max_combinations_per_reactant=MAX_COMBINATIONS,
    save_interval=SAVE_INTERVAL,
    random_sample_bounds=RANDOM_SAMPLE_BOUNDS,
    sampling_method=SAMPLING_METHOD,
    lhs_seed=LHS_SEED,
    save_metadata=IF_SAVE_METADATA,
    save_training_data=IF_SAVE_TRAINING_DATA,
    save_complete_csv=IF_SAVE_COMPLETE_CSV,
)

elapsed_time = time.time() - start_time
hours = int(elapsed_time // 3600)
minutes = int((elapsed_time % 3600) // 60)
seconds = int(elapsed_time % 60)

print(f"\n[OK] Data generation completed!")
print(f"  - Total time: {hours:02d}:{minutes:02d}:{seconds:02d}")

# Calculate actual number of simulations completed
# The dataset has multiple rows per simulation (one per reactor step)
# We get the actual count from the metadata file
if dataset is not None and len(dataset) > 0:
    import glob
    metadata_files = sorted(glob.glob(str(generator.output_dir / 'metadata_*.json')), reverse=True)
    if metadata_files:
        with open(metadata_files[0], 'r') as f:
            metadata = json.load(f)
        n_simulations = metadata.get('total_simulations', len(REACTANTS) * MAX_COMBINATIONS)
    else:
        n_simulations = len(REACTANTS) * MAX_COMBINATIONS
    
    print(f"  - Simulations completed: {n_simulations}")
    print(f"  - Data points collected: {len(dataset):,}")
    if n_simulations > 0:
        print(f"  - Average time per simulation: {elapsed_time / n_simulations:.2f} seconds")
        print(f"  - Data points per simulation: {len(dataset) / n_simulations:.1f}")
    else:
        print("  - Average time: N/A (no simulations completed)")
else:
    print("  - Average time: N/A (no data collected)")


> 🧠 **Concept — Failed runs are a fact of life.** In any large simulation campaign some fraction of runs will not finish: the stiff solver stalls at its minimum step size, a parameter combination turns out to be physically extreme, convergence tolerances are never met. The campaign behind this course attempted 50,000 simulations and completed 45,932 — a 91.9% success rate, which is entirely typical. The generator's job is to fail *gracefully*: record the failure together with the inputs that caused it, skip, and continue — run 4,317 must never be able to kill a 20-hour campaign. For ML, the bookkeeping matters more than the plumbing: failures are rarely uniform across the input space, and every failure is a hole in the training coverage. Keeping the failure log next to the dataset (the metadata JSON here does) is what lets you check, later, whether your surrogate's usable domain has quietly shrunk.
>
> *In your domain:* diverged CFD cases, crashed climate-ensemble members, non-converged DFT relaxations — same phenomenon, same rule: ship the failure log with the dataset.

#### 🤔 Think it through — where did the missing 8% go?

Suppose the ~4,000 failed runs are not scattered uniformly but cluster in one corner of the parameter space — say, high wall heat flux combined with low mass flow, the hottest and most aggressive cracking conditions. What happens to a surrogate trained on the survivors, and to the Bayesian optimisation in Lesson 10 that later trusts it?

<details><summary><b>💡 Discussion</b></summary>

The training data silently under-covers exactly that corner — and so does the test set, which is drawn from the same survivors, so no metric will warn you: the surrogate extrapolates there while its scores look excellent. Worse, aggressive corners are often the *interesting* ones (yields tend to peak near operating limits), so an optimiser is actively drawn to the region where the surrogate is least constrained and can return confident nonsense. Defences: plot the failure locations recorded in the metadata and look for clustering; treat "solver failed" as information — it often marks near-unphysical operating points that should be excluded from the search domain deliberately; constrain the optimiser to well-covered regions; and always re-validate any proposed optimum with the real simulator, which is exactly the Cantera check Lesson 10 performs.
</details>

#### 🤔 Think it through — nine million rows, but how many data points?

The output above reports ~200 data points per simulation; at full scale the campaign produced over 9 million rows from ~46,000 runs. When this data is split into training and test sets from Lesson 3 onwards, would a random 80/20 split over rows be acceptable? What is the effective sample size of this dataset?

<details><summary><b>💡 Discussion</b></summary>

No. The 200 rows of one run are near-siblings — identical inlet conditions, state varying smoothly along z. A random row-level split puts roughly 160 rows of each run in training and 40 in test, so the model is evaluated on close copies of examples it trained on, and the score becomes a flattering illusion (leakage). The unit of statistical independence is the run, not the row: the effective sample size is closer to 46,000 than to 9.2 million. Every split later in the course is therefore made at *run* level — a run's rows all go to train or all to test. This is the single most common trap when ML practitioners first meet simulation, time-series or patient-level data, and it should change how you read every metric in the rest of the course.
</details>